# Pick and Place with the 2D Two-Link Arm

This notebook hands the `TwoLinkArmWithObject2D` example to a `SteppingPlanner` with a single goal — "block at this placement point" — and lets the planner figure out the rest. Internally the planner does a BFS over modes:

1. From the initial mode (block pinned to world), it can't reach the placement goal directly.
2. It tries the bundled `pickup_transition`. The transition's trigger is "tip at block", which the planner *can* reach — it plans into that state and applies the transition.
3. In the resulting mode (block rigidly attached to the arm) the placement goal *is* reachable; the planner finishes the path.

We never tell the planner about the pickup. The single `planner.plan(...)` call returns a complete trajectory.

In [1]:
import numpy as np
from IPython.display import HTML
from matplotlib import animation, pyplot
from spatialmath import SE2

from comb.constraints import ConstraintParameters, PointEquality2D
from comb.examples.two_link_arm_with_object_2d import TwoLinkArmWithObject2D
from comb.planners.stepping import SteppingPlanner
from comb.rendering.matplotlib_2d import MatplotlibRenderer2D
from comb.rendering.overlays import PointMarker2D

## 1. Build the scene

`TwoLinkArmWithObject2D` packages: a two-link arm, an anchored world body, a `block` body initially pinned to the world via a `FixedJoint2D`, and a `pickup_transition` (bundled in `ex.system.transitions`) that fires when the arm's tip coincides with the block.

In [2]:
ex = TwoLinkArmWithObject2D(block_pose=SE2(0.4, 1.4, 0.0))
block_pickup_xy = (
    float(ex.mode.body_poses[ex.block].t[0]),
    float(ex.mode.body_poses[ex.block].t[1]),
)
placement_xy = (-0.6, 1.2)
print(f"Block starts at {block_pickup_xy}, will be placed at {placement_xy}")

Block starts at (0.4, 1.4), will be placed at (-0.6, 1.2)


## 2. Define the goal and plan

The goal is just a `PointEquality2D` pinning `ex.block` to the placement world point. Hand `ex.system` (which already has the pickup transition bundled) and the goal to `SteppingPlanner.plan`. One call returns a complete `Trajectory[ModeState]`.

In [3]:
placement_goal = PointEquality2D(
    body1=ex.world,
    body2=ex.block,
    fixed_parameters=ConstraintParameters(
        values=np.array([placement_xy[0], placement_xy[1], 0.0, 0.0]),
        names=PointEquality2D.fixed_parameter_names(),
    ),
)

planner = SteppingPlanner(interval=0.1)
trajectory = planner.plan(ex.system, [placement_goal], horizon=2.0)

end_state = trajectory(trajectory.duration)
print(f"Trajectory duration: {trajectory.duration} s")
print(f"Final block position: {tuple(end_state.body_poses[ex.block].t)}")

Trajectory duration: 2.0000000000000013 s
Final block position: (np.float64(-0.6000000000584174), np.float64(1.1999999999126694))


## 3. Animate

Sample the trajectory at fixed intervals, push each `ModeState` into `ex.mode` so the renderer picks up the right poses, and render. Two `PointMarker2D` overlays show the pickup point (a hollow gray ring so the block at the start doesn't occlude it) and the placement target (a star).

In [4]:
fig, ax = pyplot.subplots(figsize=(5, 5))
renderer = MatplotlibRenderer2D(ax=ax, xlim=(-2.0, 2.0), ylim=(-1.0, 2.5))

samples = list(trajectory.enumerate(0.05))
overlays = [
    PointMarker2D(
        x=block_pickup_xy[0],
        y=block_pickup_xy[1],
        marker="o",
        color="tab:gray",
        size=400.0,
        filled=False,
    ),
    PointMarker2D(
        x=placement_xy[0], y=placement_xy[1], marker="*", color="tab:orange", size=300.0
    ),
]


def draw(frame_idx: int):
    _, state = samples[frame_idx]
    for body in ex.mode.bodies:
        ex.mode.body_poses[body] = state.body_poses[body]
    for c in ex.mode.configuration:
        ex.mode.configuration[c] = state.configuration[c]
    renderer.render(ex.mode, overlays=overlays)
    return []


anim = animation.FuncAnimation(fig, draw, frames=len(samples), interval=50)
pyplot.close(fig)
HTML(anim.to_jshtml())

## What to try next

- Move `placement_xy` somewhere harder to reach (still inside `2 * link_length`) and replay.
- Tighten / loosen the planner's `interval` and observe the trajectory smoothness.
- Drop the trigger tolerance (`pickup_tolerance` in the example's constructor) and see how close the planner has to get before the pickup transition fires.